# 01 · Análisis exploratorio de la OBT MC1

**Objetivo.** Entender la OBT que dejó el PSet #2 antes de modelar, y tomar con evidencia cuatro
decisiones que usan los notebooks siguientes:

1. qué variante de representación usar (`R` = valor crudo, `N` = normalizado o `RN` = ambas);
2. cuál es el "mejor atributo SMART" para el baseline de regla del PSet #1;
3. cómo tratar colas largas, outliers y nulos en el `Pipeline`;
4. qué esquema de validación exige la estructura temporal del problema.

**Datos.** Snapshot Parquet de `OBT_MC1_RN` extraído de Snowflake (ver `manifiesto.json`). Toda la EDA usa
**solo el período de entrenamiento/CV (2018-01 → 2019-09)**: octubre de 2019 es embargo y noviembre de 2019 es la
prueba única del notebook 05, así que no se miran aquí.

**Unidad y métrica.** Fila = SSD-día. La métrica del proyecto es **por evento** (PSet #1): una alerta por disco, que
es acierto si el disco falla entre 7 y 30 días después. Por eso la potencia univariada de cada atributo se mide
con la PR-AUC por evento de `src/pipeline.py`, no con métricas por fila.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl_
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import datos, pipeline as pl

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")

DATOS = Path(os.environ.get("DATOS_DIR", "/datos")) / "obt_mc1"
FIG = Path("../docs/figuras"); FIG.mkdir(parents=True, exist_ok=True)
SEMILLA = pl.SEMILLA
rng = np.random.default_rng(SEMILLA)
T0 = time.time()

manifiesto = datos.leer_manifiesto(DATOS)
pd.Series({k: v for k, v in manifiesto.items() if k != "archivos"}, name="snapshot").to_frame()

## 1. Tamaño, tipos y nulos

El esquema y los nulos se leen de los metadatos de los Parquet (cada *row group* guarda su `null_count`), así se
cubren las 841 columnas sin cargarlas a memoria.

In [ ]:
archivos_train = [p for p in datos.archivos_snapshot(DATOS)
                  if p.parent.name.split("=")[1] <= datos.FIN_ENTRENAMIENTO.strftime("%Y-%m")]
esquema = pq.read_schema(archivos_train[0])
nulos = dict.fromkeys(esquema.names, 0); filas_train = 0
for p in archivos_train:
    md_ = pq.ParquetFile(p).metadata
    filas_train += md_.num_rows
    for rg in range(md_.num_row_groups):
        g = md_.row_group(rg)
        for c in range(g.num_columns):
            col = g.column(c)
            nulos[col.path_in_schema] += col.statistics.null_count
nulos = pd.Series(nulos) / filas_train

tipos = pd.Series({n: str(esquema.field(n).type) for n in esquema.names})
print(f"Snapshot completo: {manifiesto['filas']:,} filas · {len(esquema.names)} columnas")
print(f"Entrenamiento/CV (2018-01 → 2019-09): {filas_train:,} filas en {len(archivos_train)} archivos")
tipos.value_counts().rename("columnas").to_frame()

In [ ]:
def familia_columna(c):
    if c in datos.COLUMNAS_META or c == datos.FEATURE_EDAD:
        return c
    rep, attr, *resto = c.split("_")
    return f"{rep}_x" + ("_" + "_".join(resto) if resto else " (valor del día)")

def atributo(c):
    partes = c.split("_")
    return int(partes[1]) if partes[0] in ("R", "N") and partes[1].isdigit() else None

resumen_nulos = (nulos.rename("pct_nulos").to_frame()
                 .assign(familia=lambda d: d.index.map(familia_columna), atributo=lambda d: d.index.map(atributo)))
por_atributo = (resumen_nulos.dropna(subset=["atributo"])
                .assign(rep=lambda d: d.index.str[0])
                .pivot_table(index="atributo", columns="rep", values="pct_nulos", aggfunc="mean"))
dia = pd.DataFrame({"R_dia": [nulos[f"R_{a}"] for a in datos.SMART_IDS],
                    "N_dia": [nulos[f"N_{a}"] for a in datos.SMART_IDS]}, index=list(datos.SMART_IDS))
tabla_nulos = dia.join(por_atributo.add_suffix("_promedio_38_cols")).mul(100).round(3)
tabla_nulos.index.name = "atributo SMART"
tabla_nulos.sort_values("R_dia", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
tabla_nulos["R_dia"].sort_values(ascending=False).plot.bar(ax=ax, color="#4C72B0")
ax.set_ylabel("% de filas nulas"); ax.set_xlabel("atributo SMART (valor crudo del día)")
ax.set_title("Nulos por atributo en entrenamiento/CV")
fig.tight_layout(); fig.savefig(FIG / "01_nulos_por_atributo.png", dpi=150); plt.show()
print("Columnas de metadatos sin nulos:", {c: f"{nulos[c]:.4%}" for c in datos.COLUMNAS_META})

## 2. Carga del período de entrenamiento

Se cargan las columnas que usa la EDA: metadatos, la edad observada (`DAYS_SINCE_FIRST_SEEN`), los valores del día
`R_x` y `N_x` de los 22 atributos, y para cada atributo crudo tres features temporales representativas
(`MEAN_30D`, `MEAN_SHIFT_7D`, `MEAN_SHIFT_30D`). `cargar_dataset` verifica antes el SHA-256 del snapshot.

Con 100M filas, la EDA trabaja en **polars** (multihilo, `SSD_KEY` categórica): las agregaciones sobre todo el
período corren en segundos. pandas se usa solo sobre una muestra de 5M filas para correlaciones y gráficos.

In [ ]:
R_DIA = [f"R_{a}" for a in datos.SMART_IDS]
N_DIA = [f"N_{a}" for a in datos.SMART_IDS]
TEMPORALES = [f"R_{a}_{s}" for a in datos.SMART_IDS for s in ("MEAN_30D", "MEAN_SHIFT_7D", "MEAN_SHIFT_30D")]

t = time.time()
train = datos.cargar_dataset(DATOS, [datos.FEATURE_EDAD] + R_DIA + N_DIA + TEMPORALES,
                             hasta=datos.FIN_ENTRENAMIENTO, como="polars")
assert train["OBSERVATION_DATE"].max() <= datos.FIN_ENTRENAMIENTO
assert train.height == filas_train
print(f"{train.height:,} filas × {train.width} columnas · {train.estimated_size('gb'):.1f} GB · {time.time()-t:.0f} s")

# Arreglos compactos para las métricas por evento (SSD como entero, fecha como día).
SSD = train["SSD_KEY"].to_physical().to_numpy().astype(np.int64)
FECHA = train["OBSERVATION_DATE"].cast(pl_.Int32).to_numpy()   # días desde 1970
DIAS = train["DAYS_TO_FAILURE"].to_numpy()
Y = train["TARGET_30D"].to_numpy()
col = pl_.col

## 3. Variable objetivo y desbalance

In [ ]:
# Fecha de falla de cada disco que falla: fecha + DAYS_TO_FAILURE en cualquiera de sus filas positivas.
eventos = (train.filter(col("TARGET_30D") == 1)
           .group_by("SSD_KEY")
           .agg((col("OBSERVATION_DATE") + pl_.duration(days=col("DAYS_TO_FAILURE").cast(pl_.Int64))).min().alias("falla")))
n_discos = train["SSD_KEY"].n_unique()
anticipables = pl.eventos_anticipables(SSD, DIAS)
desbalance = pd.Series({
    "filas": train.height,
    "filas positivas (TARGET_30D=1)": int(Y.sum()),
    "prevalencia por fila": Y.mean(),
    "negativas por cada positiva": (1 - Y.mean()) / Y.mean(),
    "discos (SSD) distintos": n_discos,
    "discos que fallan (eventos)": eventos.height,
    "eventos anticipables (≥1 obs. a 7–30 días)": anticipables,
    "prevalencia por disco": eventos.height / n_discos,
    "días observados por disco (mediana)": train.group_by("SSD_KEY").len()["len"].median(),
}, name="entrenamiento/CV").to_frame()
desbalance

In [ ]:
por_mes = (train.group_by(col("OBSERVATION_DATE").dt.truncate("1mo").alias("mes"))
           .agg(pl_.len().alias("filas"), (col("TARGET_30D").mean() * 100).alias("prevalencia_%"),
                col("SSD_KEY").n_unique().alias("discos_activos"))
           .join(eventos.group_by(col("falla").dt.truncate("1mo").alias("mes")).agg(pl_.len().alias("eventos_falla")),
                 on="mes", how="left")      # fallas de oct-2019 (embargo) no se grafican
           .sort("mes").fill_null(0)
           .with_columns((col("eventos_falla") / col("discos_activos") * 1000).alias("eventos_por_1000_discos"))
           .to_pandas().set_index("mes"))
por_mes.index = por_mes.index.to_period("M")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
x = por_mes.index.astype(str)
axes[0].bar(x, por_mes["prevalencia_%"], color="#C44E52"); axes[0].set_title("Prevalencia por fila (%)")
axes[1].bar(x, por_mes["eventos_falla"], color="#55A868"); axes[1].set_title("Eventos de falla por mes")
axes[2].plot(x, por_mes["discos_activos"] / 1e3, marker="o"); axes[2].set_title("Discos activos (miles)")
for ax in axes:
    ax.tick_params(axis="x", rotation=90, labelsize=8)
fig.tight_layout(); fig.savefig(FIG / "01_target_por_mes.png", dpi=150); plt.show()
por_mes.round(3)

In [ ]:
semestre = (col("OBSERVATION_DATE").dt.year().cast(pl_.String) + "-S"
            + pl_.when(col("OBSERVATION_DATE").dt.month() <= 6).then(pl_.lit("1")).otherwise(pl_.lit("2")))
deriva = (train.group_by(semestre.alias("semestre"))
          .agg((col("TARGET_30D").mean() * 100).alias("prevalencia_%"),
               col(datos.FEATURE_EDAD).median().alias("edad_mediana_dias"),
               ((col("R_5") > 0).mean() * 100).alias("pct_filas_R5>0"),
               col("R_9").median().alias("R_9_mediana (horas encendido)"))
          .sort("semestre").to_pandas().set_index("semestre"))
deriva.round(3)

## 4. Estadística descriptiva

Valores crudos del día (`R_x`). Se agregan la asimetría y el porcentaje de ceros, que son los que deciden la
transformación (`log1p`) y el tratamiento de outliers.

In [ ]:
muestra = train.sample(n=min(5_000_000, train.height), seed=SEMILLA).to_pandas()  # para correlaciones y gráficos
q = [.5, .9, .99, .999]
filas = []
for c in R_DIA:  # cada estadística sobre las ~100M filas, en polars
    e = train.select(col(c).count().alias("count"), col(c).mean().alias("mean"), col(c).std().alias("std"),
                     col(c).min().alias("min"), *[col(c).quantile(p).alias(f"{p:.1%}") for p in q],
                     col(c).max().alias("max"), ((col(c) == 0).mean() * 100).alias("pct_ceros"),
                     col(c).skew().alias("asimetria"), col(c).kurtosis().alias("curtosis"),
                     col(c).n_unique().alias("n_valores_distintos")).to_pandas()
    filas.append(e.assign(atributo=c))
desc = pd.concat(filas).set_index("atributo")
desc.round(2)

## 5. Outliers: IQR y Z-score

Para cada atributo se marca como outlier (a) lo que cae fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]` y (b) lo que tiene
`|z| ≥ 3`. Lo que importa para el modelado no es solo cuántos hay, sino **si los outliers concentran las fallas**:
por eso se calcula la prevalencia dentro y fuera de los outliers (lift).

In [ ]:
filas = []
for c in R_DIA:
    v = train[c].to_numpy().astype(np.float64); ok = ~np.isnan(v); vv, yy = v[ok], Y[ok]
    q1, q3 = np.percentile(vv, [25, 75]); iqr = q3 - q1
    out_iqr = (vv < q1 - 1.5 * iqr) | (vv > q3 + 1.5 * iqr)
    out_z = np.abs((vv - vv.mean()) / vv.std()) >= 3 if vv.std() > 0 else np.zeros_like(vv, bool)
    prev = yy.mean()
    filas.append({
        "atributo": c, "Q1": q1, "Q3": q3, "IQR": iqr,
        "pct_outlier_IQR": out_iqr.mean() * 100, "pct_outlier_Z3": out_z.mean() * 100,
        "lift_outlier_IQR": (yy[out_iqr].mean() / prev) if out_iqr.any() else np.nan,
        "lift_outlier_Z3": (yy[out_z].mean() / prev) if out_z.any() else np.nan,
    })
outliers = pd.DataFrame(filas).set_index("atributo")
outliers.sort_values("lift_outlier_IQR", ascending=False).round(3)

In [ ]:
top_lift = outliers["lift_outlier_IQR"].sort_values(ascending=False).index[:8].tolist()
idx_pos = np.flatnonzero(Y == 1)
idx_neg = rng.choice(np.flatnonzero(Y == 0), size=500_000, replace=False)
caja = train.select(top_lift + ["TARGET_30D"])[np.concatenate([idx_pos, idx_neg])].to_pandas()
largo = caja.melt(id_vars="TARGET_30D", var_name="atributo", value_name="valor").dropna()
largo["log1p(valor)"] = np.sign(largo["valor"]) * np.log1p(np.abs(largo["valor"]))

fig, ax = plt.subplots(figsize=(13, 4.2))
sns.boxplot(data=largo, x="atributo", y="log1p(valor)", hue="TARGET_30D", ax=ax, fliersize=1,
            palette={0: "#8DA0CB", 1: "#E78AC3"})
ax.set_title("Atributos con mayor lift de outliers: distribución por clase (todas las positivas + 500k negativas)")
fig.tight_layout(); fig.savefig(FIG / "01_boxplot_outliers.png", dpi=150); plt.show()

## 6. Correlaciones: Pearson vs. Spearman

**Por qué Spearman.** Los atributos SMART son contadores o medidas monótonas, con una masa enorme en cero y
colas largas (sección 4). Pearson mide relación *lineal* y lo dominan unos pocos valores extremos; Spearman
trabaja con rangos, así que captura relaciones monótonas no lineales ("más errores ⇒ más riesgo") y es robusto a
las colas. Se calculan ambos para mostrar la diferencia. Se usa una muestra aleatoria simple de 5M filas (≈ 6 %
del período; la estimación de un coeficiente con 5M filas tiene error estándar < 0,001).

In [ ]:
cols_corr = R_DIA + [c for c in TEMPORALES if c.endswith("MEAN_SHIFT_30D")] + [datos.FEATURE_EDAD]
pearson = muestra[cols_corr].corrwith(muestra["TARGET_30D"], method="pearson")
rangos = muestra[cols_corr + ["TARGET_30D"]].rank()
spearman = rangos[cols_corr].corrwith(rangos["TARGET_30D"])
corr_target = pd.DataFrame({"pearson": pearson, "spearman": spearman})
corr_target["|spearman| - |pearson|"] = corr_target["spearman"].abs() - corr_target["pearson"].abs()
corr_target.sort_values("spearman", key=np.abs, ascending=False).round(4).head(25)

In [ ]:
spear_attr = rangos[R_DIA].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(spear_attr, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={"shrink": .7, "label": "Spearman"})
ax.set_title("Spearman entre atributos SMART (valor crudo del día, muestra de 5M filas)")
fig.tight_layout(); fig.savefig(FIG / "01_heatmap_spearman.png", dpi=150); plt.show()

pares = (spear_attr.where(np.triu(np.ones(spear_attr.shape, bool), k=1)).stack()
         .rename("spearman").to_frame().query("abs(spearman) >= 0.8").sort_values("spearman", key=np.abs, ascending=False))
pares

## 7. Definición del episodio de alerta

El PSet #1 define la métrica **por evento**: una alerta abre un *episodio*, y el episodio es acierto si el disco
falla entre 7 y 30 días después. La primera implementación permitía **un solo episodio por disco en todo el
historial**. La tabla siguiente muestra por qué esa lectura literal no sirve: con reglas de umbral simples, los
discos que alertan sí fallan mucho más que el promedio, pero su **primera** alerta llega en general meses antes
de la falla y con la lectura literal el disco queda como falsa alarma y como falla perdida a la vez.

**Decisión (equipo):** un episodio dura 30 días (el horizonte del target) y al terminar el disco puede volver a
alertar, como ocurre en operación (la alerta es un ticket: se revisa el disco y vuelve a servicio). La versión
literal se conserva como análisis de sensibilidad (`episodio_dias=None`).

In [ ]:
falla_disco = (train.filter(col("DAYS_TO_FAILURE").is_not_null()).group_by("SSD_KEY")
               .agg((col("OBSERVATION_DATE") + pl_.duration(days=col("DAYS_TO_FAILURE").cast(pl_.Int64))).min().alias("falla")))
filas = []
for c, q in (("R_5", 0.99), ("R_5", 0.999), ("R_187", 0.999), ("R_173", 0.99)):
    umbral = train[c].quantile(q)
    primera = (train.filter(col(c) >= umbral).group_by("SSD_KEY").agg(col("OBSERVATION_DATE").min().alias("alerta"))
               .join(falla_disco, on="SSD_KEY", how="left"))
    d = primera.select((col("falla") - col("alerta")).dt.total_days().alias("d")).to_pandas()["d"]
    filas.append({"regla": f"{c} ≥ {umbral:g} (p{q*100:g})", "discos que alertan": len(primera),
                  "de ellos fallan después": int(d.notna().sum()),
                  "% que falla": d.notna().mean() * 100,
                  "1ª alerta a 7–30 d (acierto literal)": int(d.between(7, 30).sum()),
                  "1ª alerta a >30 d": int((d > 30).sum()),
                  "mediana de anticipación si >30 d": d[d > 30].median()})
print(f"Referencia: {falla_disco.height / n_discos:.2%} de los discos falla en el período")
pd.DataFrame(filas).set_index("regla").round(1)

## 8. Potencia univariada por evento

Para cada columna se calcula la **PR-AUC por evento** (episodios de 30 días) usando la columna misma como score,
en los dos sentidos (valor alto = riesgo y valor bajo = riesgo), sobre todo el período de entrenamiento. Es la
métrica de selección del proyecto, así que responde dos preguntas: cuál es el mejor atributo para la regla de
umbral y si las features temporales aportan señal sobre el valor del día. El nivel de azar es la PR-AUC media de
tres scores aleatorios.

In [ ]:
PREP = pl.preparar_eventos(SSD, FECHA, DIAS)  # se ordena una vez y se reutiliza en todas las curvas

def pr_auc_univariada(x, signo, episodio_dias=pl.EPISODIO_DIAS):
    x = np.asarray(x, dtype=np.float64)
    s = np.where(np.isnan(x), -np.inf, signo * x)  # sin dato -> nunca alerta
    return pl.curva_evento(None, None, None, s, episodio_dias=episodio_dias, prep=PREP).pr_auc

cols_auc = R_DIA + N_DIA + TEMPORALES + [datos.FEATURE_EDAD]
tareas = [(c, s) for c in cols_auc for s in (1, -1)]
t = time.time()
res = [pr_auc_univariada(train[c].to_numpy(), s) for c, s in tareas]
azar = float(np.mean([pr_auc_univariada(rng.random(len(train)), 1) for _ in range(3)]))
print(f"{len(tareas)} curvas en {time.time()-t:.0f} s · PR-AUC por evento de un score aleatorio: {azar:.4f}")

auc = pd.DataFrame(tareas, columns=["columna", "signo"]).assign(pr_auc_evento=res)
mejor = auc.loc[auc.groupby("columna")["pr_auc_evento"].idxmax()].set_index("columna")
mejor["sentido"] = np.where(mejor["signo"] > 0, "alto = riesgo", "bajo = riesgo")
mejor["lift_vs_azar"] = mejor["pr_auc_evento"] / azar
mejor.loc[R_DIA + N_DIA].sort_values("pr_auc_evento", ascending=False).round(4).head(15)

In [ ]:
# Sensibilidad: misma métrica con la lectura literal (un solo episodio por disco)
literal = pd.Series({c: pr_auc_univariada(train[c].to_numpy(), int(mejor.loc[c, "signo"]), episodio_dias=None)
                     for c in R_DIA}, name="pr_auc_literal")
azar_literal = float(np.mean([pr_auc_univariada(rng.random(len(train)), 1, None) for _ in range(3)]))
sens = mejor.loc[R_DIA, ["pr_auc_evento", "sentido"]].join(literal)
sens["lift_episodios"] = sens["pr_auc_evento"] / azar
sens["lift_literal"] = sens["pr_auc_literal"] / azar_literal
print(f"Azar: episodios {azar:.4f} · literal {azar_literal:.4f}")
sens.sort_values("pr_auc_evento", ascending=False).round(4)

### 8.1 R vs. N: ¿aporta algo la versión normalizada?

In [ ]:
r_vs_n = pd.DataFrame({
    "pr_auc_R": mejor.loc[R_DIA, "pr_auc_evento"].to_numpy(),
    "pr_auc_N": mejor.loc[N_DIA, "pr_auc_evento"].to_numpy(),
    "spearman_R_N": [rangos_rn for rangos_rn in
                     (muestra[[f"R_{a}", f"N_{a}"]].corr(method="spearman").iloc[0, 1] for a in datos.SMART_IDS)],
    "nulos_R_%": [nulos[f"R_{a}"] * 100 for a in datos.SMART_IDS],
    "nulos_N_%": [nulos[f"N_{a}"] * 100 for a in datos.SMART_IDS],
    "distintos_R": [muestra[f"R_{a}"].nunique() for a in datos.SMART_IDS],
    "distintos_N": [muestra[f"N_{a}"].nunique() for a in datos.SMART_IDS],
}, index=pd.Index(datos.SMART_IDS, name="atributo"))
r_vs_n["ganador"] = np.where(r_vs_n["pr_auc_R"] >= r_vs_n["pr_auc_N"], "R", "N")
print("Atributos donde R ≥ N en PR-AUC por evento:", (r_vs_n["ganador"] == "R").sum(), "de", len(r_vs_n))
print("Mediana de |Spearman(R, N)|:", r_vs_n["spearman_R_N"].abs().median().round(3))
r_vs_n.round(4)

### 8.2 ¿Las features temporales aportan sobre el valor del día?

In [ ]:
temporal = pd.DataFrame({
    "dia": mejor.loc[R_DIA, "pr_auc_evento"].to_numpy(),
    "MEAN_30D": mejor.loc[[f"R_{a}_MEAN_30D" for a in datos.SMART_IDS], "pr_auc_evento"].to_numpy(),
    "MEAN_SHIFT_7D": mejor.loc[[f"R_{a}_MEAN_SHIFT_7D" for a in datos.SMART_IDS], "pr_auc_evento"].to_numpy(),
    "MEAN_SHIFT_30D": mejor.loc[[f"R_{a}_MEAN_SHIFT_30D" for a in datos.SMART_IDS], "pr_auc_evento"].to_numpy(),
}, index=pd.Index(datos.SMART_IDS, name="atributo"))
temporal["mejor_temporal"] = temporal[["MEAN_30D", "MEAN_SHIFT_7D", "MEAN_SHIFT_30D"]].max(axis=1)
temporal["ganancia_vs_dia"] = temporal["mejor_temporal"] - temporal["dia"]
temporal = temporal.sort_values("dia", ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
temporal[["dia", "MEAN_30D", "MEAN_SHIFT_7D", "MEAN_SHIFT_30D"]].head(12).plot.bar(ax=ax, width=.8)
ax.axhline(azar, color="k", ls="--", lw=1, label="azar")
ax.set_ylabel("PR-AUC por evento (univariada)"); ax.set_title("Valor del día vs. features temporales (12 mejores atributos)")
ax.legend(ncols=5, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "01_temporal_vs_dia.png", dpi=150); plt.show()
temporal.round(4)

In [ ]:
edad = mejor.loc[datos.FEATURE_EDAD]
print(f"{datos.FEATURE_EDAD}: PR-AUC por evento {edad['pr_auc_evento']:.4f} ({edad['sentido']}), "
      f"lift {edad['lift_vs_azar']:.1f}× sobre azar")
mejor_r = mejor.loc[R_DIA, "pr_auc_evento"].idxmax()
print("Mejor atributo crudo para la regla:", mejor_r, mejor.loc[mejor_r, ["pr_auc_evento", "sentido"]].to_dict())

## 9. Decisiones que salen de la EDA

In [ ]:
signo_regla = int(mejor.loc[mejor_r, "signo"])
decisiones = {
    "variante": "R",
    "atributo_regla": mejor_r,
    "sentido_regla": mejor.loc[mejor_r, "sentido"],
    "pr_auc_evento_regla_univariada": round(float(mejor.loc[mejor_r, "pr_auc_evento"]), 4),
    "pr_auc_evento_azar": round(float(azar), 4),
    "episodio_dias": pl.EPISODIO_DIAS,
    "eventos_anticipables_train": int(anticipables),
    "prevalencia_fila_train": round(float(Y.mean()), 6),
    "dataset_sha256": manifiesto["dataset_sha256"],
}
Path("../docs/eda_decisiones.json").write_text(json.dumps(decisiones, indent=2, ensure_ascii=False))
mejor.round(5).to_csv("../docs/eda_pr_auc_univariada.csv")
print(f"Tiempo total del notebook: {(time.time()-T0)/60:.1f} min")
decisiones